# 🧬 Extra — reshaping and joining

**Extra material.** Nothing in the assessed work assumes this notebook. It is the step that loses most people, so it is here on its own, to be taken slowly and come back to.

**Time:** ~40 minutes &nbsp;•&nbsp; **Goal:** change the *shape* of a table, and bring in information that is not in the file.

Two operations, and between them they cover most of what is left:

- **`melt`** — turn eight insect columns into two: *which insect* and *how many*
- **`merge`** — attach a second table, matching on a shared key

## Run this first

In [ ]:
# Run this first. Everything sessions 1 and 2 taught you, in one cell.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

df.columns = df.columns.str.strip()                      # the header bug
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
df["Date"] = pd.to_datetime(df["Date"])                  # real dates
df["Month"] = pd.Categorical(                            # months in season order
    df["Date"].dt.month_name(),
    categories=["April", "May", "June", "July", "August", "September"],
    ordered=True,
)
df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]
df["All_Insects"] = df[INSECTS].sum(axis=1)

import matplotlib.pyplot as plt

SPECIES = ["Terrestris", "Lapidarius", "Pascuorum",
           "Pratorum", "Hypnorum", "Hortorum"]

print(df.shape)
df.head(3)

## Skill 1 — Why the shape is wrong

Look at what you have been writing all lab:

```python
df.groupby("Crop")["Bumblebees"].sum()
df.groupby("Crop")["Honeybees"].sum()
df.groupby("Crop")["Hoverflies"].sum()
```

Three near-identical lines, because **which insect** is stored in the *column names* rather than in the data. You cannot group by a column name. So you cannot write "total visits, per crop, per insect" in one line — even though it is one question.

This is **wide** format. It is the right shape for a clipboard: one row per watch, tick the boxes. It is the wrong shape for analysis.

**Long** (or *tidy*) format puts one observation per row:

| Crop | Insect | Count |
|---|---|---|
| Raspberry | Honeybees | 2 |
| Raspberry | Bumblebees | 0 |
| Raspberry | Hoverflies | 1 |

Eight times as many rows, and every question becomes one `groupby`.

## Skill 2 — `melt`

```python
long = df.melt(
    id_vars=["Date", "Allotment", "Crop", "Flowers"],   # keep these as they are
    value_vars=INSECTS,                                 # unpivot these
    var_name="Insect",                                  # the old column names go here
    value_name="Count",                                 # the values go here
)
```

Four arguments, and only two ideas:

- `id_vars` — the columns that **identify** a row. They get repeated, once per melted column.
- `value_vars` — the columns that get **folded down** into two new ones, named by `var_name` and `value_name`.

Everything not named in either is dropped. That is the most common surprise; name what you need to keep.

In [ ]:
# DEMO - eight columns become two
long = df.melt(
    id_vars=["Date", "Year", "Month", "Allotment", "Community", "Crop", "Flowers"],
    value_vars=INSECTS,
    var_name="Insect",
    value_name="Count",
)

print("wide:", df.shape)
print("long:", long.shape, " -- 1620 x 8 =", 1620 * 8)
long.head(10)

In [ ]:
# DEMO - and now the question that needed three lines takes one
per_crop_insect = (
    long.groupby(["Crop", "Insect"])["Count"]
    .sum()
    .unstack()
)
per_crop_insect.loc[["Raspberry", "Squash", "Broad bean", "Apple"]]

### 🧬 DIY challenge

1. Melt again, this time keeping only `Month` and `Crop` as `id_vars`. How many columns does the result have?
2. From `long`, total visits per insect per **month**, in calendar order.
3. `long` still has the `Flowers` column, repeated eight times per watch. What would go wrong if you ran `long["Flowers"].sum()`? What is the right way to get the total flower count from `long`?

> 💡 Question 3 is the real hazard of long format: anything belonging to the *watch* rather than the *observation* is now duplicated. Sum it and you get eight times too much.

In [ ]:
# TODO - your turn

## Skill 3 — Use the new shape to check the old one

The file contains six bumblebee species columns **and** a `Bumblebees` total. Two columns that should agree, which means you can check one against the other — the habit from session 2.

Long format makes the check a one-liner.

In [ ]:
# DEMO - do the six species add up to the total?
species_total = df[SPECIES].sum(axis=1)

print("rows where the species do not sum to Bumblebees:",
      (species_total != df["Bumblebees"]).sum())
print()
print("total from the six species:", species_total.sum())
print("total in the Bumblebees column:", df["Bumblebees"].sum())

Zero mismatches, on all 1,620 rows. That is worth doing even when — especially when — you expect it to pass: it tells you the species columns are complete and that `Bumblebees` is a derived total rather than a separate, sloppier count.

Now melt the species and ask which one is actually doing the work.

In [ ]:
# DEMO - six species, through the season
sp_long = df.melt(
    id_vars=["Month", "Crop", "Flowers"],
    value_vars=SPECIES,
    var_name="Species",
    value_name="Count",
)

by_species = sp_long.groupby("Species")["Count"].sum().sort_values(ascending=False)
print(by_species)

fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)
by_species.sort_values().plot(kind="barh", ax=ax, color="#5fb37a")
ax.set_xlabel("total visits recorded")
ax.set_title("Bombus species on Brighton allotments")
plt.show()

### 🧬 DIY challenge

1. Which species is most common, and what fraction of all bumblebee visits is it?
2. Group `sp_long` by `Species` and `Month`, unstack, and look for a species whose season is different from the others.
3. `Hypnorum` (the tree bumblebee) only arrived in Britain in 2001. How many visits did it get here?

> 💡 The species columns are *not* mutually exclusive with `Bumblebees` — melting both together would double-count. Melt one set or the other, never both.

In [ ]:
# TODO - your turn

## Skill 4 — `pivot_table` goes back the other way

`pivot_table` is melt's inverse: it takes long data and spreads one column back out into many.

```python
long.pivot_table(
    index="Crop",         # rows
    columns="Insect",     # one new column per value found here
    values="Count",
    aggfunc="sum",        # what to do when several rows land in the same cell
)
```

The `aggfunc` is the part people miss. A pivot has to combine everything that falls into each cell, so it is a `groupby` with a nicer shape — which is exactly what `.groupby([a, b]).sum().unstack()` was doing in notebook 5. Same result, fewer moving parts.

Use `groupby` to compute, `pivot_table` to present.

In [ ]:
# DEMO - a presentation table, in one call
table = long.pivot_table(
    index="Crop",
    columns="Insect",
    values="Count",
    aggfunc="sum",
)
table.loc[["Raspberry", "Squash", "Broad bean", "Runner bean", "Apple"],
          ["Honeybees", "Bumblebees", "Hoverflies", "Beetles"]]

### 🧬 DIY challenge

1. Pivot with `index="Month"` and `columns="Community"`, summing `Count`. Add `margins=True` and see what appears.
2. Change `aggfunc` to `"mean"`. What does each cell mean now, and why is it so much less useful here?
3. Rebuild the same table with `groupby(...).sum().unstack()` and confirm the numbers match.

> 💡 If a pivot comes out full of `NaN`, it usually means that combination never happened — not that something broke.

In [ ]:
# TODO - your turn

## Skill 5 — `merge`: information the file does not contain 🏆

`Rbean`, `Bbean`, `Physalis`. The survey records what was watched, but nothing about **what kind of flower it is** — and that is what would actually explain the results.

So put it in a second file, and join the two on their shared column:

```python
crops = pd.read_csv("crops.csv")
m = df.merge(crops, on="Crop_Generic", how="left")
```

| `how=` | keeps |
|---|---|
| `"inner"` | only rows that matched in **both** — the default |
| `"left"` | every row of the left table, `NaN` where no match |
| `"outer"` | everything from both |

**Always check the row count afterwards.** A merge that silently drops rows (`inner`, no match) or silently multiplies them (a key that is not unique) is the most expensive bug in this lab, because the result still looks perfectly reasonable.

In [ ]:
# DEMO - bring in the botany
LOOKUP = "crops.csv"
if not Path(LOOKUP).exists():
    urllib.request.urlretrieve(DATA_URL + LOOKUP, LOOKUP)

crops_lookup = pd.read_csv(LOOKUP)
crops_lookup["Crop_Generic"] = crops_lookup["Crop_Generic"].replace(
    {"Rbean": "Runner bean", "Bbean": "Broad bean"})

print(crops_lookup.head())

m = df.merge(crops_lookup, left_on="Crop", right_on="Crop_Generic", how="left")

print()
print("rows before:", len(df), " after:", len(m))
print("unmatched crops:", m[m["Family"].isna()]["Crop"].unique())

In [ ]:
# DEMO - and now a question the survey could not answer on its own
shape = m.groupby("Flower_Shape").agg(
    watches=("Bumblebees", "size"),
    bumblebees=("Bumblebees", "sum"),
    honeybees=("Honeybees", "sum"),
    flowers=("Flowers", "sum"),
)
shape["bb_rate"] = shape["bumblebees"] / shape["flowers"]
shape["hb_rate"] = shape["honeybees"] / shape["flowers"]

shape[["watches", "bb_rate", "hb_rate"]].round(4).sort_values("bb_rate", ascending=False)

`bell` flowers — currant, gooseberry, blueberry — get almost no visits per flower from either bee, and they flower in April when little is flying. `tube` flowers, the cucurbits, get the highest rate of both, and honeybees work them five times as hard as bumblebees do.

None of that was in the survey file. It came from eighteen rows of botany you attached in one line.

### 🏆 DIY finale — a joined analysis

1. Merge the lookup and build a per-`Family` summary: watches, flowers, and the per-flower rate for bumblebees, honeybees and hoverflies.
2. Apply an n-guard — families watched fewer than 50 times are not worth ranking.
3. Draw it: one figure, families on the y-axis, per-flower rate on the x-axis, one colour per insect. (`shape[["bb_rate", "hb_rate"]].plot(kind="barh", ax=ax)` draws a whole DataFrame as grouped bars.)
4. The `Insect_Dependence` column says how much each crop *needs* insects. Do the crops that need pollinators most actually get the most visits per flower? Write two sentences on what you find.

Question 4 is a real research question, and the answer in this dataset is not a tidy one. Say what you see, including the parts that do not fit.

> 💡 Plotting a DataFrame rather than a Series gives you one set of bars per column, with a legend for free. That is why getting the table into the right shape first is most of the work.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Reshaping and pivot tables, with diagrams: <https://pandas.pydata.org/docs/user_guide/reshaping.html>
- Merging, joining and concatenating — the `how=` diagrams are worth printing out: <https://pandas.pydata.org/docs/user_guide/merging.html>
- `pd.concat([a, b])` stacks tables that have the same columns — the other way of combining, when you have more rows rather than more columns.
- `df.merge(..., validate="many_to_one")` makes pandas **check** that the key is unique on the right-hand side, and raise if it is not. Use it every time.

## Recap

You can now:

- tell **wide** from **long**, and say why groupby wants long
- reshape with `melt(id_vars=..., value_vars=...)` and spot the duplicated columns it creates
- check one set of columns against another, and use long format to do it
- present a result with `pivot_table(index=, columns=, values=, aggfunc=)`
- join a second table with `merge(on=, how=)` — and **always check the row count afterwards**

That is the pandas toolkit. Everything else you will meet is a variation on these.